# Module 6 · Mini-project and report
**Xenoi Biotech · DNA-Seq (Variant Analysis) Workshop** · Week 4 · Days 26–30

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjalianjali1234/dna-seq-variant-colab/blob/main/notebooks/M6_Mini_project_and_report.ipynb)

**By the end of this module:** you have run a complete, reproducible variant-analysis study and written it up as a short scientific report.

**Workshop question:** can different variant callers recover the same small DNA changes in a real human sample (HG001 / NA12878), and how do we know which calls to trust?

**How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code under it. Cells marked ✍️ are exercises: write your answers in your lab notebook.

## Session setup (run every time you open Colab)
Colab gives you a fresh computer each session, so this cell installs the tools and downloads the data. The first run takes about **3–5 minutes**; go through the reading below while it works.

In [ ]:
import os
%cd /content
if not os.path.exists("dna-seq-variant-colab"):
    !git clone -q https://github.com/anjalianjali1234/dna-seq-variant-colab.git
%cd /content/dna-seq-variant-colab
!bash setup/colab_setup.sh
os.environ["PATH"] = "/content/env/bin:" + os.environ["PATH"]
!samtools --version | head -1 && gatk --version 2>/dev/null | head -1

### Catch up on earlier modules
This rebuilds the outputs of Modules 1–3 (about 2–4 minutes). Skip it if you ran them in this same session.

In [ ]:
!bash setup/catchup.sh 3

> 💾 **Optional:** to keep your outputs between sessions, mount Google Drive and copy `run/` there at the end:
> `from google.colab import drive; drive.mount('/content/drive')` then `!cp -r run /content/drive/MyDrive/dna-seq-run`

## Day 26 · Project question and plan
**Research question:** how do stricter quality and depth filters change precision and recall for GATK, BCFtools and FreeBayes on HG001?

Mean depth in this region is about 47×, so we test **DP ≥ 10, 20 and 40**: the strictest threshold removes real variants, giving a genuine trade-off to explain.

✍️ Before running, write down your prediction: how will precision and recall move as DP rises?

## Day 27 · Run the experiment

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
for dp in 10 20 40; do python scripts/benchmark.py 30 $dp > /dev/null; done
python - <<'PY'
import pandas as pd, glob
df = pd.concat(pd.read_csv(f, sep="\t") for f in sorted(glob.glob("run/report/benchmark_q30_dp*.tsv")))
df.to_csv("run/report/benchmark_all.tsv", sep="\t", index=False); print(df.to_string(index=False))
PY

## Day 28 · Figures

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv("run/report/benchmark_all.tsv", sep="\t")
df["precision"] = pd.to_numeric(df.precision, errors="coerce")
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for metric, a in zip(["precision", "recall"], ax):
    for c, g in df.groupby("caller"):
        a.plot(g.min_dp, g[metric], marker="o", label=c)
    a.set(title=metric.capitalize(), xlabel="minimum read depth (DP)", xticks=[10, 20, 40], ylim=(0, 1.05))
ax[0].set_ylabel("score"); ax[1].legend()
fig.suptitle("HG001 · chr20:10,000,001–10,010,000 · GRCh37 · QUAL ≥ 30")
plt.tight_layout(); plt.savefig("run/report/metrics.png", dpi=200); plt.show()

## Day 29 · Review one discrepancy
Pick one false positive, false negative or indel from `alleles_q30_dp40.tsv` and review it in IGV (Module 5, Day 24). Explain whether the difference comes from the reads, the caller, the filter or how the allele is written.

In [ ]:
import pandas as pd
a = pd.read_csv("run/report/alleles_q30_dp40.tsv", sep="\t")
a[(a.benchmark == "match") & (a[["gatk", "bcftools", "freebayes"]].sum(axis=1) < 3)].head(10)

## Day 30 · Write the report
Write 500–800 words with these sections, then download `run/report/` (benchmark tables, `metrics.png`, annotated table) as part of your submission.

1. **Title, author, date, build and region**
2. **Question and prediction**
3. **Methods**: data (HG001, NIST truth set), QC, alignment (BWA-MEM2, Picard), callers and versions, filters, benchmark rule
4. **Results**: QC and coverage numbers, benchmark table, figure, annotation summary
5. **Evidence at two loci**: IGV observations and decisions
6. **Discussion**: what surprised you, limitations (small masked region, one sample), and one follow-up experiment

| Assessment | Points |
|---|---:|
| Reproducibility: versions, commands, data sources | 4 |
| Correct comparison design | 4 |
| Results and figure | 5 |
| Read-level reasoning (IGV) | 4 |
| Interpretation and limitations | 3 |

In [ ]:
!zip -qr /content/my_variant_report.zip run/report
from google.colab import files
files.download("/content/my_variant_report.zip")